# 03 · Modelling street crime
**Is street light availability a useful predictor of street crime?**

We follow the steps from the ML seminars: a linear regression, SHAP explanations,
a decision tree and a random forest, then cross-validation, `GridSearchCV` and a
final test.

Run the notebook once with `target = 'street_rate'`, then change it to
`'comparison_rate'` and run it again. Comparing the two runs tells us whether
lighting matters more for street crime.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shap

In [ ]:
study = gpd.read_file('../data/processed/study_area.gpkg')

target = 'street_rate'          # change to 'comparison_rate' for the second run

## Section 1 — Inputs and target

`features` lists the inputs. Population controls for how many people live in a
square; we divide it by 1,000 to keep the numbers small (**2.5 means 2,500
residents**). If you added OpenStreetMap data in notebook 01, `n_poi` (shops,
pubs, takeaways) controls for how busy it is.

Crime rates are very skewed, so we predict `np.log1p()` of the rate.

In [ ]:
study['population_1000'] = study['population'] / 1000

features = ['slai', 'population_1000']
if 'n_poi' in study.columns:
    features.append('n_poi')

X = study[features]
y = np.log1p(study[target])
features

Set aside 20% of the squares for testing. `random_state=42` makes the split repeatable.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Training squares:', len(X_train))
print('Test squares:', len(X_test))

## Section 2 — Linear regression

`fit()` trains the model on the training squares.

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

`coef_` gives one coefficient per input, in the order of `features`. Holding the
other inputs fixed, the SLAI coefficient is the change in log crime rate for a
one-unit change in SLAI (from the least to the most lit square).

In [ ]:
coefficients = pd.Series(linear_model.coef_, index=features)
coefficients

Predict the test squares and measure the errors. Lower **RMSE** is better;
**R²** of 1 is perfect and 0 is no better than predicting the mean.

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

linear_test_predictions = linear_model.predict(X_test)
print('Linear RMSE:', root_mean_squared_error(y_test, linear_test_predictions))
print('Linear R²:', r2_score(y_test, linear_test_predictions))

### SHAP for the linear model

**Baseline prediction + contributions = the square's predicted value.**
We use 100 squares as the reference for the baseline.

In [ ]:
background = X_train.sample(100, random_state=42)

explainer = shap.LinearExplainer(linear_model, background)
shap_values = explainer(X_test)

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.beeswarm(shap_values)

## Section 3 — Decision tree and random forest

A small tree (3 levels) shows the rules it learns. It is for interpretation;
we do not expect it to be the most accurate model.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

tree_model = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

plt.figure(figsize=(18, 7))
plot_tree(tree_model, feature_names=features, impurity=False, rounded=True, fontsize=9)
plt.tight_layout()
plt.show()

A **random forest** combines 100 trees. The steps are the same: create, fit, predict.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(n_estimators=100, random_state=42)
forest_model.fit(X_train, y_train)

forest_test_predictions = forest_model.predict(X_test)
print('Forest RMSE:', root_mean_squared_error(y_test, forest_test_predictions))
print('Forest R²:', r2_score(y_test, forest_test_predictions))

`feature_importances_` summarises how much the forest uses each input. It does
not tell us whether an input raises or lowers crime, or that it causes it.

In [ ]:
importance = pd.Series(forest_model.feature_importances_, index=features)
importance.sort_values().plot.barh(figsize=(6, 3))
plt.xlabel('Feature importance in this fitted forest')
plt.tight_layout()
plt.show()

## Section 4 — Choosing the forest's settings with cross-validation

Fitting and checking on the same squares hides overfitting. **Cross-validation**
splits the training squares into folds: each fold is held out once while the
model is fitted on the rest.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV

cv = KFold(n_splits=5, shuffle=True, random_state=42)

`GridSearchCV` tries each `max_depth` and keeps the one with the lowest
cross-validated RMSE. It uses only the training squares.

In [ ]:
forest_params = {'max_depth': [2, 4, 8, None]}

forest_search = GridSearchCV(
    RandomForestRegressor(n_estimators=100, random_state=42), forest_params,
    cv=cv, scoring='neg_root_mean_squared_error'
)
forest_search.fit(X_train, y_train)
forest_search.best_params_

In [ ]:
print('Best cross-validated RMSE:', -forest_search.best_score_)

### Do neighbouring squares make the scores look too good?

Neighbouring squares are similar, so a random split can put near-copies in both
the training and the test folds. **GroupKFold** keeps each 10 km block together,
so the model is always tested on blocks it has not seen.

In [ ]:
from sklearn.model_selection import GroupKFold

best_forest = forest_search.best_estimator_
blocks_train = study.loc[X_train.index, 'block']

random_scores = -cross_val_score(best_forest, X_train, y_train, cv=cv,
                                 scoring='neg_root_mean_squared_error')
spatial_scores = -cross_val_score(best_forest, X_train, y_train, cv=GroupKFold(n_splits=5),
                                  groups=blocks_train, scoring='neg_root_mean_squared_error')
print('Random folds RMSE: ', random_scores.mean())
print('Spatial folds RMSE:', spatial_scores.mean())

If the spatial RMSE is higher, random folds were giving an optimistic picture
of how well the model predicts places it has not seen.

## Section 5 — Final test

Use the chosen forest once on the test squares.

In [ ]:
final_model = best_forest
pred_test = final_model.predict(X_test)

print('Final RMSE:', root_mean_squared_error(y_test, pred_test))
print('Final R²:', r2_score(y_test, pred_test))

In [ ]:
plt.scatter(y_test, pred_test, s=10, alpha=0.5)
plt.plot([0, y.max()], [0, y.max()], color='black', linestyle='--')
plt.xlabel('Observed (log rate)')
plt.ylabel('Predicted (log rate)')
plt.title('Random forest: test squares')
plt.show()

### SHAP for the random forest

`TreeExplainer` is the version of SHAP for tree models. Where do high SLAI
values (red dots) fall: do they push predictions up or down?

In [ ]:
tree_explainer = shap.TreeExplainer(final_model)
forest_shap = tree_explainer(X_test)
shap.plots.beeswarm(forest_shap)

## Section 6 — Where are the errors?

A **residual** is observed minus predicted. Red = the model predicts too much
crime; blue = too little. Clusters of one colour suggest something the inputs miss.

In [ ]:
test_map = study.loc[X_test.index].copy()
test_map['residual'] = y_test - pred_test

ax = study.plot(color='lightgrey', figsize=(7, 7))
test_map.plot(ax=ax, column='residual', cmap='RdBu', vmin=-1.5, vmax=1.5, legend=True)
plt.title('Random forest residuals (test squares)')
plt.axis('off')
plt.show()

## Section 7 — Save the results for this target

The results table gets the target in its name, so the two runs do not overwrite each other.

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear regression', 'Random forest'],
    'Test RMSE': [root_mean_squared_error(y_test, linear_test_predictions),
                  root_mean_squared_error(y_test, pred_test)],
    'Test R²': [r2_score(y_test, linear_test_predictions), r2_score(y_test, pred_test)],
    'SLAI coefficient': [coefficients['slai'], None],
    'Mean |SHAP| of SLAI': [np.abs(shap_values[:, 'slai'].values).mean(),
                            np.abs(forest_shap[:, 'slai'].values).mean()],
})
results.to_csv(f'../outputs/tables/results_{target}.csv', index=False)
results

**Now change `target` to `'comparison_rate'` and run the notebook again.**
If the SLAI matters about as much for comparison crime as for street crime,
lighting is probably standing in for busy, densely populated places rather than
affecting crime directly.